## 面试问题

循环历史摘要压缩：何时触发、压缩什么、不丢什么？

## 回答主线

历史太长时用摘要压缩，把旧的多步压成结构化摘要。触发是 token 逼近预算；压缩冗余中间步；绝不能压掉关键事实（订单号、金额、约束）。本 Notebook 用 10 步退款历史，对比结构化好摘要（保留 order_id/amount/verified）与自由式坏摘要（丢金额），展示坏摘要导致下游无法退款。

## 真实案例

10 步历史含关键事实 `order_id=od-77`、`amount=88`、`verified=True` 与大量冗余中间日志。压缩成「结构化摘要 + 最近 2 步」。数据为教学历史，不代表真实系统。

In [1]:
history = []  # 构造一段 10 步退款历史。
history.append({"step": 1, "kind": "fact", "text": "order_id=od-77"})  # 关键事实：订单号。
history.append({"step": 2, "kind": "fact", "text": "amount=88"})  # 关键事实：金额。
for i in range(3, 10):  # 生成一批冗余中间步。
    history.append({"step": i, "kind": "log", "text": f"中间日志 {i}"})  # 冗余中间日志。
history.append({"step": 10, "kind": "fact", "text": "verified=True"})  # 关键事实：已校验。

print("历史步数:", len(history))  # 展示历史长度。
print("关键事实步:", [h["text"] for h in history if h["kind"] == "fact"])  # 展示其中的关键事实。

历史步数: 10
关键事实步: ['order_id=od-77', 'amount=88', 'verified=True']


## 基线（Baseline）

反面基线：直接截断只保留最近 2 步。它把订单号和金额都丢在前面，下游根本看不到。

In [2]:
def truncate_tail(history, k=2):  # 截断策略：只保留最近 k 步。
    return history[-k:]  # 返回末尾 k 步。

tail = truncate_tail(history, k=2)  # 截断保留最近 2 步。
tail_facts = [h["text"] for h in tail if h["kind"] == "fact"]  # 抽取截断后可见的关键事实。
print("截断保留:", [h["text"] for h in tail])  # 展示截断丢掉了订单号和金额。
print("截断可见关键事实:", tail_facts)  # 展示关键事实大量丢失。

截断保留: ['中间日志 9', 'verified=True']
截断可见关键事实: ['verified=True']


## 失败案例与修正

坏摘要是「自由总结一句话」，把金额总结没了，下游无法退款；好摘要是结构化抽取所有 `kind==fact` 的关键事实并保留最近 2 步原文。这就是「结构化抽取优于自由总结」。

In [3]:
def summarize_structured(history):  # 好摘要：结构化抽取所有关键事实。
    facts = [h["text"] for h in history if h["kind"] == "fact"]  # 只抽取标注为 fact 的关键信息。
    recent = history[-2:]  # 保留最近 2 步原文。
    return {"facts": facts, "recent": [h["text"] for h in recent]}  # 返回结构化摘要。

def summarize_freeform(history):  # 坏摘要：自由总结只留一句概述。
    return {"note": "处理了一笔退款，做了若干校验", "recent": [history[-1]["text"]]}  # 丢掉了订单号和金额。

good_summary = summarize_structured(history)  # 生成结构化好摘要。
bad_summary = summarize_freeform(history)  # 生成自由式坏摘要。
print("好摘要事实:", good_summary["facts"])  # 展示好摘要保留全部关键事实。
print("坏摘要内容:", bad_summary)  # 展示坏摘要丢掉金额与订单号。

好摘要事实: ['order_id=od-77', 'amount=88', 'verified=True']
坏摘要内容: {'note': '处理了一笔退款，做了若干校验', 'recent': ['verified=True']}


In [4]:
def issue_refund(summary):  # 依据摘要发起退款，需要订单号和金额。
    text = " ".join(summary.get("facts", []))  # 拼接摘要中的事实。
    has_order = "order_id=" in text  # 检查是否保留订单号。
    has_amount = "amount=" in text  # 检查是否保留金额。
    if has_order and has_amount:  # 两个关键事实齐备才能退款。
        return "refund_issued"  # 成功发起退款。
    return "missing_key_facts"  # 关键事实缺失无法退款。

good_result = issue_refund(good_summary)  # 用好摘要发起退款。
bad_result = issue_refund(bad_summary)  # 用坏摘要发起退款。
print("好摘要退款结果:", good_result)  # 展示保留关键事实可退款。
print("坏摘要退款结果:", bad_result)  # 展示丢金额导致无法退款。

好摘要退款结果: refund_issued
坏摘要退款结果: missing_key_facts


## 结果解读

结构化好摘要保留 order_id/amount/verified，下游成功退款；自由式坏摘要丢掉金额，下游报「关键事实缺失」。压缩必须先定义摘要必含字段，并延续「事实/推测」标注，还要保留可回取指针以便需要细节时回取。

In [5]:
print("好摘要保留金额:", any("amount=" in f for f in good_summary["facts"]))  # 好摘要保留金额。
print("坏摘要保留金额:", "amount" in str(bad_summary))  # 坏摘要丢掉金额。
print("退款结果 好 vs 坏:", good_result, "vs", bad_result)  # 展示摘要质量决定下游成败。

好摘要保留金额: True
坏摘要保留金额: False
退款结果 好 vs 坏: refund_issued vs missing_key_facts


In [6]:
assert "amount=88" in good_summary["facts"]  # 好摘要必须保留金额。
assert "order_id=od-77" in good_summary["facts"]  # 好摘要必须保留订单号。
assert good_result == "refund_issued"  # 好摘要下游可成功退款。
assert bad_result == "missing_key_facts"  # 坏摘要丢关键事实导致失败。
assert len(good_summary["facts"]) == 3  # 好摘要抽取到三条关键事实。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
